# XMAiNframe — COBOL Inference Tests

> **GPU runtime required.** Runtime → Change runtime type → T4 GPU (free tier) or A100 (Colab Pro)

Tests [XMAiNframe-instruct-7b](https://huggingface.co/Fsoft-AIC/XMAiNframe-instruct-7b) across three evaluation tasks:

| Task | Description |
|------|-------------|
| Summarization | Explain what a COBOL program does |
| Question Answering | Answer specific questions about COBOL code |
| Multiple Choice | COBOL / mainframe knowledge questions |

## 1 · Environment Check & Install

In [1]:
import sys, os
import psutil

try:
    import torch
    has_cuda = torch.cuda.is_available()
except ImportError:
    has_cuda = False

mem = psutil.virtual_memory()

print("=" * 55)
print("  HARDWARE CHECK")
print("=" * 55)
print(f"  RAM total    : {mem.total/1024**3:.1f} GB")
print(f"  RAM available: {mem.available/1024**3:.1f} GB")

if has_cuda:
    props = torch.cuda.get_device_properties(0)
    vram  = props.total_memory / 1024**3
    print(f"  GPU          : {props.name}")
    print(f"  VRAM         : {vram:.1f} GB")
    if vram >= 14:
        print("  float16 inference: OK")
    elif vram >= 4:
        print("  4-bit quantized inference: OK (bitsandbytes)")
    else:
        print("  GPU VRAM too low — switch to a T4 runtime")
else:
    print("  No GPU detected — go to Runtime > Change runtime type > T4 GPU")
print("=" * 55)

  HARDWARE CHECK
  RAM total    : 12.7 GB
  RAM available: 11.4 GB
  No GPU detected — go to Runtime > Change runtime type > T4 GPU


In [2]:
!pip install -q transformers accelerate bitsandbytes sentencepiece
print("Packages installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 9.6 MB/s eta 0:00:00:00:0100:01m
Packages installed.


## 2 · COBOL Test Programs

In [3]:
COBOL_HELLO = """\
      *================================================================*
      * PROGRAM-ID: HELLO
      * PURPOSE: Basic COBOL program — fundamental structure & syntax.
      *================================================================*
       IDENTIFICATION DIVISION.
       PROGRAM-ID. HELLO.

       ENVIRONMENT DIVISION.
       CONFIGURATION SECTION.
       SOURCE-COMPUTER. IBM-MAINFRAME.
       OBJECT-COMPUTER. IBM-MAINFRAME.

       DATA DIVISION.
       WORKING-STORAGE SECTION.
       01  WS-NAME           PIC X(30) VALUE 'XMAiNframe User'.
       01  WS-DATE.
           05 WS-YEAR        PIC 9(4).
           05 WS-MONTH       PIC 9(2).
           05 WS-DAY         PIC 9(2).
       01  WS-FORMATTED-DATE PIC X(10).

       PROCEDURE DIVISION.
       MAIN-PARAGRAPH.
           ACCEPT WS-DATE FROM DATE YYYYMMDD
           STRING WS-YEAR DELIMITED BY SIZE
                  '-'      DELIMITED BY SIZE
                  WS-MONTH DELIMITED BY SIZE
                  '-'      DELIMITED BY SIZE
                  WS-DAY   DELIMITED BY SIZE
                  INTO WS-FORMATTED-DATE
           END-STRING
           DISPLAY 'Hello, ' WS-NAME '!'
           DISPLAY 'Today is: ' WS-FORMATTED-DATE
           STOP RUN.
"""

COBOL_CUSTMGMT = """\
      *================================================================*
      * PROGRAM-ID: CUSTMGMT
      * PURPOSE: Customer management — file I/O, record processing,
      *          error handling, VSAM-style indexed file operations.
      *================================================================*
       IDENTIFICATION DIVISION.
       PROGRAM-ID. CUSTMGMT.

       ENVIRONMENT DIVISION.
       INPUT-OUTPUT SECTION.
       FILE-CONTROL.
           SELECT CUSTOMER-FILE
               ASSIGN TO 'CUSTFILE'
               ORGANIZATION IS INDEXED
               ACCESS MODE IS DYNAMIC
               RECORD KEY IS CUST-ID
               ALTERNATE RECORD KEY IS CUST-NAME
                   WITH DUPLICATES
               FILE STATUS IS WS-FILE-STATUS.
           SELECT REPORT-FILE
               ASSIGN TO 'CUSTRPT'
               ORGANIZATION IS SEQUENTIAL
               FILE STATUS IS WS-RPT-STATUS.

       DATA DIVISION.
       FILE SECTION.
       FD  CUSTOMER-FILE.
       01  CUSTOMER-RECORD.
           05 CUST-ID            PIC 9(8).
           05 CUST-NAME          PIC X(40).
           05 CUST-BALANCE       PIC S9(7)V99 COMP-3.
           05 CUST-CREDIT-LIMIT  PIC S9(7)V99 COMP-3.
           05 CUST-STATUS        PIC X(1).
              88 CUST-ACTIVE     VALUE 'A'.
              88 CUST-INACTIVE   VALUE 'I'.
              88 CUST-SUSPENDED  VALUE 'S'.

       FD  REPORT-FILE.
       01  REPORT-RECORD         PIC X(132).

       WORKING-STORAGE SECTION.
       01  WS-FILE-STATUS        PIC X(2).
           88 WS-SUCCESS         VALUE '00'.
           88 WS-EOF             VALUE '10'.
       01  WS-COUNTERS.
           05 WS-READ-COUNT      PIC 9(6) VALUE ZEROS.
           05 WS-ACTIVE-COUNT    PIC 9(6) VALUE ZEROS.
           05 WS-INACTIVE-COUNT  PIC 9(6) VALUE ZEROS.
           05 WS-HIGH-BAL-COUNT  PIC 9(6) VALUE ZEROS.
       01  WS-TOTALS.
           05 WS-TOTAL-BALANCE   PIC S9(11)V99 VALUE ZEROS.
           05 WS-MAX-BALANCE     PIC S9(7)V99  VALUE ZEROS.
       01  WS-CREDIT-THRESHOLD   PIC S9(7)V99 VALUE +5000.00.
       01  WS-DETAIL-LINE.
           05 WS-DET-ID          PIC 9(8).
           05 FILLER             PIC X(2) VALUE SPACES.
           05 WS-DET-NAME        PIC X(40).
           05 WS-DET-STATUS      PIC X(10).
           05 WS-DET-BALANCE     PIC Z,ZZZ,ZZ9.99-.
           05 WS-DET-FLAG        PIC X(15).

       PROCEDURE DIVISION.
       0000-MAIN-CONTROL.
           PERFORM 1000-INITIALIZE
           PERFORM 2000-PROCESS-CUSTOMERS
           PERFORM 3000-GENERATE-SUMMARY
           PERFORM 9000-CLEANUP
           STOP RUN.

       2000-PROCESS-CUSTOMERS.
           READ CUSTOMER-FILE NEXT RECORD
               AT END SET WS-EOF TO TRUE
           END-READ
           PERFORM UNTIL WS-EOF
               ADD 1 TO WS-READ-COUNT
               PERFORM 2100-EVALUATE-CUSTOMER
               PERFORM 2200-UPDATE-STATISTICS
               PERFORM 2300-WRITE-DETAIL
               READ CUSTOMER-FILE NEXT RECORD
                   AT END SET WS-EOF TO TRUE
               END-READ
           END-PERFORM.

       2100-EVALUATE-CUSTOMER.
           EVALUATE TRUE
               WHEN CUST-ACTIVE
                   ADD 1 TO WS-ACTIVE-COUNT
                   MOVE 'ACTIVE' TO WS-DET-STATUS
               WHEN CUST-INACTIVE
                   ADD 1 TO WS-INACTIVE-COUNT
                   MOVE 'INACTIVE' TO WS-DET-STATUS
               WHEN OTHER
                   MOVE 'UNKNOWN' TO WS-DET-STATUS
           END-EVALUATE.

       2200-UPDATE-STATISTICS.
           ADD CUST-BALANCE TO WS-TOTAL-BALANCE
           IF CUST-BALANCE > WS-MAX-BALANCE
               MOVE CUST-BALANCE TO WS-MAX-BALANCE
           END-IF
           IF CUST-BALANCE > WS-CREDIT-THRESHOLD
               ADD 1 TO WS-HIGH-BAL-COUNT
           END-IF.

       2300-WRITE-DETAIL.
           MOVE CUST-ID      TO WS-DET-ID
           MOVE CUST-NAME    TO WS-DET-NAME
           MOVE CUST-BALANCE TO WS-DET-BALANCE
           IF CUST-BALANCE > CUST-CREDIT-LIMIT
               MOVE 'OVER LIMIT' TO WS-DET-FLAG
           ELSE IF CUST-BALANCE > WS-CREDIT-THRESHOLD
               MOVE 'HIGH BALANCE' TO WS-DET-FLAG
           ELSE
               MOVE SPACES TO WS-DET-FLAG
           END-IF
           WRITE REPORT-RECORD FROM WS-DETAIL-LINE.

       3000-GENERATE-SUMMARY.
           DISPLAY 'Total Records: ' WS-READ-COUNT
           DISPLAY 'Active:        ' WS-ACTIVE-COUNT
           DISPLAY 'High Balance:  ' WS-HIGH-BAL-COUNT
           DISPLAY 'Total Balance: ' WS-TOTAL-BALANCE
           DISPLAY 'Max Balance:   ' WS-MAX-BALANCE.
"""

COBOL_PAYROLL = """\
      *================================================================*
      * PROGRAM-ID: PAYROLL
      * PURPOSE: Payroll processor — PERFORM VARYING, tax bracket
      *          tables, OCCURS, COMPUTE, department accumulators.
      *================================================================*
       IDENTIFICATION DIVISION.
       PROGRAM-ID. PAYROLL.

       DATA DIVISION.
       WORKING-STORAGE SECTION.
       01  WS-TAX-TABLE.
           05 WS-BRACKET OCCURS 7 TIMES.
              10 WS-BRACKET-LOW    PIC 9(9)V99.
              10 WS-BRACKET-HIGH   PIC 9(9)V99.
              10 WS-BRACKET-RATE   PIC 9V99.

       01  WS-CALC-FIELDS.
           05 WS-GROSS-PAY        PIC S9(7)V99 VALUE ZEROS.
           05 WS-OVERTIME-PAY     PIC S9(7)V99 VALUE ZEROS.
           05 WS-FEDERAL-TAX      PIC S9(7)V99 VALUE ZEROS.
           05 WS-STATE-TAX        PIC S9(7)V99 VALUE ZEROS.
           05 WS-FICA-TAX         PIC S9(7)V99 VALUE ZEROS.
           05 WS-NET-PAY          PIC S9(7)V99 VALUE ZEROS.
           05 WS-TAXABLE-INCOME   PIC S9(9)V99 VALUE ZEROS.

       01  WS-CONSTANTS.
           05 WS-OVERTIME-RATE    PIC S9V99 VALUE 1.50.
           05 WS-STANDARD-HOURS   PIC S9(3)V99 VALUE 40.00.
           05 WS-FICA-RATE        PIC SV9999 VALUE .0620.
           05 WS-STATE-TAX-RATE   PIC SV9999 VALUE .0500.
           05 WS-PAY-PERIODS      PIC 9(2)   VALUE 26.

       01  WS-DEPT-TABLE.
           05 WS-DEPT-ENTRY OCCURS 20 TIMES.
              10 WS-DEPT-CODE     PIC X(4).
              10 WS-DEPT-EMP-CNT  PIC 9(4)     VALUE ZEROS.
              10 WS-DEPT-TOTAL    PIC S9(9)V99 VALUE ZEROS.

       PROCEDURE DIVISION.
       2100-CALCULATE-GROSS-PAY.
           EVALUATE TRUE
               WHEN EMP-HOURLY
                   IF EMP-HOURS-WORKED > WS-STANDARD-HOURS
                       COMPUTE WS-REGULAR-PAY =
                           WS-STANDARD-HOURS * EMP-HOURLY-RATE
                       COMPUTE WS-OVERTIME-PAY =
                           (EMP-HOURS-WORKED - WS-STANDARD-HOURS)
                           * EMP-HOURLY-RATE * WS-OVERTIME-RATE
                       COMPUTE WS-GROSS-PAY =
                           WS-REGULAR-PAY + WS-OVERTIME-PAY
                   ELSE
                       COMPUTE WS-GROSS-PAY =
                           EMP-HOURS-WORKED * EMP-HOURLY-RATE
                   END-IF
               WHEN EMP-SALARIED
                   COMPUTE WS-GROSS-PAY =
                       EMP-ANNUAL-SALARY / WS-PAY-PERIODS
               WHEN EMP-COMMISSION
                   COMPUTE WS-GROSS-PAY =
                       (EMP-ANNUAL-SALARY / WS-PAY-PERIODS)
                       + (EMP-SALES-AMOUNT * EMP-COMMISSION-PCT)
           END-EVALUATE.

       2200-CALCULATE-TAXES.
           COMPUTE WS-TAXABLE-INCOME =
               WS-GROSS-PAY * WS-PAY-PERIODS
           MOVE ZEROS TO WS-FEDERAL-TAX
           PERFORM VARYING WS-SUB FROM 1 BY 1
               UNTIL WS-SUB > 7
               IF WS-TAXABLE-INCOME > WS-BRACKET-LOW(WS-SUB)
                   IF WS-TAXABLE-INCOME >= WS-BRACKET-HIGH(WS-SUB)
                       COMPUTE WS-FEDERAL-TAX =
                           WS-FEDERAL-TAX +
                           (WS-BRACKET-HIGH(WS-SUB) -
                            WS-BRACKET-LOW(WS-SUB))
                           * WS-BRACKET-RATE(WS-SUB)
                   ELSE
                       COMPUTE WS-FEDERAL-TAX =
                           WS-FEDERAL-TAX +
                           (WS-TAXABLE-INCOME -
                            WS-BRACKET-LOW(WS-SUB))
                           * WS-BRACKET-RATE(WS-SUB)
                   END-IF
               END-IF
           END-PERFORM
           COMPUTE WS-FEDERAL-TAX =
               WS-FEDERAL-TAX / WS-PAY-PERIODS
           COMPUTE WS-STATE-TAX =
               WS-GROSS-PAY * WS-STATE-TAX-RATE
           COMPUTE WS-FICA-TAX =
               WS-GROSS-PAY * WS-FICA-RATE.

       2400-CALCULATE-NET-PAY.
           COMPUTE WS-NET-PAY = WS-GROSS-PAY -
               WS-FEDERAL-TAX - WS-STATE-TAX - WS-FICA-TAX.
"""

COBOL_BANKTXN = """\
      *================================================================*
      * PROGRAM-ID: BANKTXN
      * PURPOSE: Banking transaction processor — CICS-style flow,
      *          DB2-like access, audit trails, financial controls.
      *================================================================*
       IDENTIFICATION DIVISION.
       PROGRAM-ID. BANKTXN.

       DATA DIVISION.
       WORKING-STORAGE SECTION.
       01  WS-TRANSACTION-REQUEST.
           05 WS-TXN-TYPE         PIC X(3).
              88 TXN-DEPOSIT      VALUE 'DEP'.
              88 TXN-WITHDRAW     VALUE 'WDR'.
              88 TXN-TRANSFER     VALUE 'TRF'.
              88 TXN-INQUIRY      VALUE 'INQ'.
              88 TXN-CLOSE-ACCT   VALUE 'CLS'.
           05 WS-TXN-ACCOUNT-FROM PIC 9(10).
           05 WS-TXN-ACCOUNT-TO   PIC 9(10).
           05 WS-TXN-AMOUNT       PIC S9(11)V99 COMP-3.

       01  WS-RESPONSE.
           05 WS-RESP-CODE        PIC 9(4).
              88 RESP-SUCCESS     VALUE 0000.
              88 RESP-INSUF-FUNDS VALUE 1001.
              88 RESP-ACCT-FROZEN VALUE 1002.
              88 RESP-ACCT-CLOSED VALUE 1003.
              88 RESP-DAILY-LIMIT VALUE 1004.
              88 RESP-INVALID-TXN VALUE 2001.
              88 RESP-ACCT-NOT-FND VALUE 2002.
              88 RESP-DB-ERROR    VALUE 9001.
           05 WS-RESP-MESSAGE     PIC X(80).
           05 WS-RESP-NEW-BALANCE PIC S9(11)V99 COMP-3.

       PROCEDURE DIVISION.
       1000-PROCESS-TRANSACTION.
           PERFORM 1100-VALIDATE-REQUEST
           IF RESP-SUCCESS
               EVALUATE TRUE
                   WHEN TXN-DEPOSIT
                       PERFORM 2000-PROCESS-DEPOSIT
                   WHEN TXN-WITHDRAW
                       PERFORM 3000-PROCESS-WITHDRAWAL
                   WHEN TXN-TRANSFER
                       PERFORM 4000-PROCESS-TRANSFER
                   WHEN TXN-INQUIRY
                       PERFORM 5000-PROCESS-INQUIRY
                   WHEN TXN-CLOSE-ACCT
                       PERFORM 6000-PROCESS-CLOSE-ACCOUNT
               END-EVALUATE
           END-IF
           PERFORM 8000-WRITE-AUDIT-TRAIL.

       3000-PROCESS-WITHDRAWAL.
           ADD WS-TXN-AMOUNT TO WS-ACCT-DAILY-USED
           IF WS-ACCT-DAILY-USED > WS-ACCT-DAILY-LIMIT
               SET RESP-DAILY-LIMIT TO TRUE
               MOVE 'DAILY WITHDRAWAL LIMIT EXCEEDED'
                   TO WS-RESP-MESSAGE
           ELSE
               COMPUTE WS-AFTER-BALANCE =
                   WS-ACCT-BALANCE - WS-TXN-AMOUNT
               IF WS-AFTER-BALANCE < (0 - WS-ACCT-OD-LIMIT)
                   SET RESP-INSUF-FUNDS TO TRUE
                   MOVE 'INSUFFICIENT FUNDS' TO WS-RESP-MESSAGE
               ELSE
                   MOVE WS-AFTER-BALANCE TO WS-ACCT-BALANCE
                   PERFORM 7000-DB-UPDATE
                   SET RESP-SUCCESS TO TRUE
               END-IF
           END-IF.

       4000-PROCESS-TRANSFER.
           COMPUTE WS-AFTER-BALANCE =
               WS-ACCT-BALANCE - WS-TXN-AMOUNT
           IF WS-AFTER-BALANCE < (0 - WS-ACCT-OD-LIMIT)
               SET RESP-INSUF-FUNDS TO TRUE
           ELSE
               MOVE WS-AFTER-BALANCE TO WS-ACCT-BALANCE
               PERFORM 7000-DB-UPDATE
               MOVE WS-TXN-ACCOUNT-TO TO WS-ACCT-NUMBER
               PERFORM 7000-DB-SELECT
               ADD WS-TXN-AMOUNT TO WS-ACCT-BALANCE
               PERFORM 7000-DB-UPDATE
               SET RESP-SUCCESS TO TRUE
           END-IF.
"""

COBOL_BATCHJCL = """\
      *================================================================*
      * PROGRAM-ID: BATCHJCL
      * PURPOSE: JCL-driven batch processor — ACCEPT/DISPLAY,
      *          RETURN-CODE, header/detail/trailer pattern,
      *          sequence checking, hash total validation.
      *================================================================*
       IDENTIFICATION DIVISION.
       PROGRAM-ID. BATCHJCL.

       DATA DIVISION.
       WORKING-STORAGE SECTION.
       01  WS-CONTROL-FIELDS.
           05 WS-PARM-DATA        PIC X(100).
           05 WS-RETURN-CODE      PIC S9(4) COMP VALUE ZEROS.

       01  WS-COUNTERS.
           05 WS-INPUT-COUNT      PIC 9(8) VALUE ZEROS.
           05 WS-OUTPUT-COUNT     PIC 9(8) VALUE ZEROS.
           05 WS-HEADER-COUNT     PIC 9(4) VALUE ZEROS.
           05 WS-DETAIL-COUNT     PIC 9(8) VALUE ZEROS.
           05 WS-TRAILER-COUNT    PIC 9(4) VALUE ZEROS.
           05 WS-ERROR-COUNT      PIC 9(6) VALUE ZEROS.
           05 WS-EXPECTED-COUNT   PIC 9(8) VALUE ZEROS.
           05 WS-SEQUENCE-PREV    PIC 9(6) VALUE ZEROS.

       01  WS-HASH-TOTAL          PIC 9(12) VALUE ZEROS.

       PROCEDURE DIVISION.
       0000-MAIN-CONTROL.
           PERFORM 0100-GET-PARAMETERS
           PERFORM 1000-INITIALIZE
           IF NOT WS-HAS-ERROR
               PERFORM 2000-PROCESS-INPUT
                   UNTIL WS-EOF OR WS-HAS-ERROR
               PERFORM 3000-VALIDATE-TOTALS
           END-IF
           PERFORM 4000-WRITE-SUMMARY
           PERFORM 9000-CLEANUP
           MOVE WS-RETURN-CODE TO RETURN-CODE
           STOP RUN.

       0100-GET-PARAMETERS.
           ACCEPT WS-PARM-DATA FROM COMMAND-LINE
           DISPLAY 'BATCH JOB STARTED'
           DISPLAY 'PARM DATA: ' WS-PARM-DATA.

       2200-PROCESS-DETAIL.
           IF IN-SEQUENCE-NUM <= WS-SEQUENCE-PREV
               DISPLAY 'SEQUENCE ERROR: ' IN-SEQUENCE-NUM
               ADD 1 TO WS-ERROR-COUNT
           END-IF
           MOVE IN-SEQUENCE-NUM TO WS-SEQUENCE-PREV
           ADD 1 TO WS-DETAIL-COUNT
           ADD IN-SEQUENCE-NUM TO WS-HASH-TOTAL
           CALL 'VALIDSUB' USING WS-SUBPGM-AREA
               ON EXCEPTION
                   DISPLAY 'VALIDATION SUBPROGRAM NOT FOUND'
                   ADD 1 TO WS-ERROR-COUNT
           END-CALL.

       3000-VALIDATE-TOTALS.
           IF WS-HEADER-COUNT = ZEROS
               DISPLAY 'ERROR: NO HEADER RECORD FOUND'
               MOVE 12 TO WS-RETURN-CODE
           END-IF
           IF WS-TRAILER-COUNT = ZEROS
               DISPLAY 'ERROR: NO TRAILER RECORD FOUND'
               MOVE 12 TO WS-RETURN-CODE
           END-IF
           IF WS-DETAIL-COUNT NOT = WS-EXPECTED-COUNT
               DISPLAY 'RECORD COUNT MISMATCH'
               MOVE 8 TO WS-RETURN-CODE
           END-IF
           IF WS-HASH-TOTAL NOT = WS-TRL-HASH-TOTAL
               DISPLAY 'HASH TOTAL MISMATCH'
               MOVE 8 TO WS-RETURN-CODE
           END-IF.
"""

COBOL_INVNTORY = """\
      *================================================================*
      * PROGRAM-ID: INVNTORY
      * PURPOSE: Inventory management — SEARCH/SEARCH ALL, SORT,
      *          INSPECT, STRING/UNSTRING, reorder detection.
      *================================================================*
       IDENTIFICATION DIVISION.
       PROGRAM-ID. INVNTORY.

       DATA DIVISION.
       WORKING-STORAGE SECTION.
       01  WS-WAREHOUSE-TABLE.
           05 WS-WH-ENTRY OCCURS 10 TIMES
                           ASCENDING KEY IS WS-WH-CODE
                           INDEXED BY WH-IDX.
              10 WS-WH-CODE       PIC X(4).
              10 WS-WH-ITEM-COUNT PIC 9(6) VALUE ZEROS.
              10 WS-WH-TOTAL-VAL  PIC S9(11)V99 COMP-3 VALUE ZEROS.

       01  WS-CATEGORY-TABLE.
           05 FILLER PIC X(23) VALUE 'ELCElectronics         '.
           05 FILLER PIC X(23) VALUE 'FRNFurniture           '.
           05 FILLER PIC X(23) VALUE 'CLTClothing            '.
           05 FILLER PIC X(23) VALUE 'FODFood & Beverage     '.
           05 FILLER PIC X(23) VALUE 'RAWRaw Materials       '.
       01  WS-CAT-TABLE-R REDEFINES WS-CATEGORY-TABLE.
           05 WS-CAT-ENTRY OCCURS 5 TIMES INDEXED BY CAT-IDX.
              10 WS-CAT-CODE      PIC X(3).
              10 WS-CAT-DESC      PIC X(20).

       PROCEDURE DIVISION.
       2110-UPDATE-WAREHOUSE-TOTALS.
           SET WH-IDX TO 1
           SEARCH WS-WH-ENTRY
               AT END
                   ADD 1 TO WS-WH-COUNT
                   SET WH-IDX TO WS-WH-COUNT
                   MOVE INV-WAREHOUSE TO WS-WH-CODE(WH-IDX)
               WHEN WS-WH-CODE(WH-IDX) = INV-WAREHOUSE
                   ADD 1 TO WS-WH-ITEM-COUNT(WH-IDX)
                   ADD WS-ITEM-VALUE TO WS-WH-TOTAL-VAL(WH-IDX)
           END-SEARCH.

       5100-LOOKUP-CATEGORY.
           SET CAT-IDX TO 1
           SEARCH WS-CAT-ENTRY
               AT END
                   MOVE 'UNKNOWN' TO WS-CAT-NAME
               WHEN WS-CAT-CODE(CAT-IDX) = INV-CATEGORY
                   MOVE WS-CAT-DESC(CAT-IDX) TO WS-CAT-NAME
           END-SEARCH.

       6000-PARSE-LOCATION.
           UNSTRING WS-LOCATION-STRING
               DELIMITED BY '-'
               INTO WS-P-WAREHOUSE WS-P-AISLE
                    WS-P-SHELF WS-P-BIN
               TALLYING IN WS-DELIM-COUNT
           END-UNSTRING.

       6200-INSPECT-DESCRIPTION.
           INSPECT INV-DESCRIPTION
               TALLYING WS-DELIM-COUNT FOR ALL SPACES
           INSPECT INV-DESCRIPTION
               REPLACING ALL LOW-VALUES BY SPACES.
"""

COBOL_PROGRAMS = {
    "HELLO":    COBOL_HELLO,
    "CUSTMGMT": COBOL_CUSTMGMT,
    "PAYROLL":  COBOL_PAYROLL,
    "BANKTXN":  COBOL_BANKTXN,
    "BATCHJCL": COBOL_BATCHJCL,
    "INVNTORY":  COBOL_INVNTORY,
}

for name, src in COBOL_PROGRAMS.items():
    lines = src.strip().count('\n') + 1
    print(f"  {name:<12} {lines:>3} lines")

  HELLO         34 lines
  CUSTMGMT     123 lines
  PAYROLL       95 lines
  BANKTXN       86 lines
  BATCHJCL      77 lines
  INVNTORY      64 lines


## 3 · Load Tokenizer & Model

Loads in float16 on GPUs with ≥ 14 GB VRAM (T4/A100), or falls back to 4-bit via `bitsandbytes` for smaller GPUs.

In [4]:
import torch
import time
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID   = "Fsoft-AIC/XMAiNframe-instruct-7b"
SYSTEM_MSG = "You are an expert mainframe and COBOL assistant."

has_cuda = torch.cuda.is_available()
use_4bit = False

if has_cuda:
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    use_4bit = vram_gb < 14
    print(f"GPU: {torch.cuda.get_device_properties(0).name}  ({vram_gb:.1f} GB VRAM)")
    print(f"Loading strategy: {'4-bit quantization' if use_4bit else 'float16'}")
else:
    print("No GPU found. Loading on CPU will be very slow.")

print("\nLoading tokenizer...")
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
print(f"Tokenizer loaded in {time.time()-t0:.1f}s  (vocab={tokenizer.vocab_size:,})")

print("\nLoading model...")
t0 = time.time()

if use_4bit:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
else:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16 if has_cuda else torch.float32,
        device_map="auto" if has_cuda else None,
        low_cpu_mem_usage=True,
        trust_remote_code=True,
    )

model.eval()
elapsed = time.time() - t0
params  = sum(p.numel() for p in model.parameters()) / 1e9
print(f"Model loaded in {elapsed:.1f}s  ({params:.2f}B parameters)")

No GPU found. Loading on CPU will be very slow.

Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/745 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/273 [00:00<?, ?it/s]

: 

: 

: 

## 4 · Inference Helper

In [ ]:
import textwrap
from IPython.display import display, Markdown

def run_inference(
    user_prompt: str,
    max_new_tokens: int = 512,
    do_sample: bool = False,
) -> str:
    messages = [
        {"from": "system", "value": SYSTEM_MSG},
        {"from": "human",  "value": user_prompt},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            top_k=50,
            top_p=0.95,
            num_return_sequences=1,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    return tokenizer.decode(
        outputs[0][len(inputs[0]):],
        skip_special_tokens=True,
    ).strip()


def show_result(task: str, program: str, question: str, answer: str, elapsed: float):
    separator = "─" * 72
    md = (
        f"### [{task}] `{program}`\n"
        f"**Prompt:** {textwrap.fill(question[:300], 90)}{'...' if len(question)>300 else ''}\n\n"
        f"**Response** *(in {elapsed:.1f}s)*:\n\n"
        + "\n".join(f"> {line}" for line in answer.splitlines()) + "\n\n"
        + separator
    )
    display(Markdown(md))

## 5 · Task 1 — COBOL Code Summarization

Ask XMAiNframe to summarize each COBOL program.

In [ ]:
summarization_tests = [
    {
        "program": "HELLO",
        "question": "Summarize the following COBOL program and explain what it does:",
        "code":    COBOL_HELLO,
    },
    {
        "program": "PAYROLL",
        "question": (
            "Provide a detailed summary of the following COBOL payroll program, "
            "describing its main logic sections, data structures, and calculations:"
        ),
        "code":    COBOL_PAYROLL,
    },
    {
        "program": "BANKTXN",
        "question": (
            "Summarize this COBOL banking transaction program. "
            "What transaction types does it support and how does error handling work?"
        ),
        "code":    COBOL_BANKTXN,
    },
]

for tc in summarization_tests:
    prompt = f"{tc['question']}\n\n{tc['code']}"
    t0     = time.time()
    answer = run_inference(prompt, max_new_tokens=512)
    show_result("Summarization", tc["program"], tc["question"], answer, time.time() - t0)

## 6 · Task 2 — Question Answering

Ask specific technical questions about the COBOL programs.

In [ ]:
qa_tests = [
    {
        "program": "CUSTMGMT",
        "question": (
            "How does the program handle the case where a customer's balance "
            "exceeds their credit limit? Refer to specific paragraph names."
        ),
        "code":    COBOL_CUSTMGMT,
    },
    {
        "program": "INVNTORY",
        "question": (
            "Explain the difference between how SEARCH and SEARCH ALL are used "
            "in this program, and why each was chosen for its specific table."
        ),
        "code":    COBOL_INVNTORY,
    },
    {
        "program": "BATCHJCL",
        "question": (
            "What validations are performed at end-of-file and what RETURN-CODE "
            "values can the program set? Explain each condition."
        ),
        "code":    COBOL_BATCHJCL,
    },
    {
        "program": "PAYROLL",
        "question": (
            "How is federal income tax calculated for each employee? "
            "Walk through the PERFORM VARYING logic step by step."
        ),
        "code":    COBOL_PAYROLL,
    },
]

for tc in qa_tests:
    prompt = (
        f"Given the following COBOL program, answer this question:\n"
        f"{tc['question']}\n\n{tc['code']}"
    )
    t0     = time.time()
    answer = run_inference(prompt, max_new_tokens=400)
    show_result("Question Answering", tc["program"], tc["question"], answer, time.time() - t0)

## 7 · Task 3 — Multiple Choice Questions

Test the model's COBOL and mainframe knowledge directly.

In [ ]:
mcq_tests = [
    {
        "id": "MCQ-01",
        "question": (
            "In a COBOL program, what is the purpose of the COMP-3 usage clause "
            "on a numeric field?\n\n"
            "A) It stores the number in EBCDIC display format\n"
            "B) It stores the number in packed decimal (BCD) format, saving storage space\n"
            "C) It stores the number as a standard binary integer\n"
            "D) It stores the number using floating-point IEEE 754 format"
        ),
        "answer": "B",
    },
    {
        "id": "MCQ-02",
        "question": (
            "In JCL-driven COBOL batch processing, what is the primary purpose of "
            "checking the RETURN-CODE (condition code) at the JCL step level?\n\n"
            "A) To get the number of records processed by the COBOL program\n"
            "B) To determine whether to execute subsequent steps based on the success "
            "or failure of the current step\n"
            "C) To estimate the elapsed CPU time of the step\n"
            "D) To set the printer output class for report generation"
        ),
        "answer": "B",
    },
    {
        "id": "MCQ-03",
        "question": (
            "What is the purpose of the 88-level condition name in COBOL?\n\n"
            "A) To define a subroutine or called program\n"
            "B) To declare a file status variable for I/O\n"
            "C) To associate a data name with specific values for conditional testing\n"
            "D) To create a pointer variable for dynamic memory"
        ),
        "answer": "C",
    },
    {
        "id": "MCQ-04",
        "question": (
            "In VSAM file processing, what does ACCESS MODE IS DYNAMIC allow?\n\n"
            "A) Only sequential access through the file\n"
            "B) Only random access by key\n"
            "C) Both sequential and random access within the same program\n"
            "D) Parallel file access across multiple COBOL programs simultaneously"
        ),
        "answer": "C",
    },
    {
        "id": "MCQ-05",
        "question": (
            "What is the key difference between GOBACK and STOP RUN in COBOL?\n\n"
            "A) They are identical in every way\n"
            "B) GOBACK returns control to the calling program or OS; STOP RUN "
            "always terminates the entire run unit\n"
            "C) GOBACK is used only in CICS online programs; STOP RUN only in batch\n"
            "D) GOBACK saves program state to disk; STOP RUN does not"
        ),
        "answer": "B",
    },
    {
        "id": "MCQ-06",
        "question": (
            "In COBOL, what does the INSPECT verb primarily do?\n\n"
            "A) Checks for file status errors after an I/O operation\n"
            "B) Validates that numeric fields contain only digits\n"
            "C) Counts and/or replaces characters or substrings within a data item\n"
            "D) Inspects the program's PROCEDURE DIVISION for syntax errors at runtime"
        ),
        "answer": "C",
    },
]

correct = 0
results = []

for tc in mcq_tests:
    prompt = (
        "Answer the following multiple-choice question about COBOL mainframes. "
        "State the letter of your chosen answer first, then briefly explain why.\n\n"
        + tc["question"]
    )
    t0      = time.time()
    answer  = run_inference(prompt, max_new_tokens=200)
    elapsed = time.time() - t0

    first_char = answer.strip()[0].upper() if answer.strip() else "?"
    is_correct = first_char == tc["answer"]
    if is_correct:
        correct += 1

    results.append({"id": tc["id"], "expected": tc["answer"],
                    "got": first_char, "correct": is_correct})
    show_result("Multiple Choice", tc["id"],
                tc["question"].split("\n")[0], answer, elapsed)

print(f"\nMCQ Score: {correct}/{len(mcq_tests)} ({correct/len(mcq_tests)*100:.0f}%)")

## 8 · Bonus — COBOL Code Generation

In [ ]:
codegen_tests = [
    {
        "id": "GEN-01",
        "prompt": (
            "Write a COBOL paragraph called CALC-INTEREST that computes monthly "
            "compound interest. Assume the following WORKING-STORAGE fields already "
            "exist:\n"
            "  WS-PRINCIPAL   PIC S9(9)V99 COMP-3\n"
            "  WS-ANNUAL-RATE PIC SV9(4) COMP-3\n"
            "  WS-MONTHS      PIC 9(3)\n"
            "  WS-RESULT      PIC S9(11)V99 COMP-3\n"
            "Use the compound interest formula: A = P * (1 + r/n)^(n*t)"
        ),
    },
    {
        "id": "GEN-02",
        "prompt": (
            "Write a COBOL WORKING-STORAGE section and PROCEDURE DIVISION paragraph "
            "to perform a linear search through a table of 50 items. Each item has:\n"
            "  - ITEM-CODE  PIC X(8)\n"
            "  - ITEM-DESC  PIC X(30)\n"
            "  - ITEM-PRICE PIC S9(5)V99 COMP-3\n"
            "Search for a given code and return the matching ITEM-PRICE, or -1 if not found."
        ),
    },
]

for tc in codegen_tests:
    t0     = time.time()
    answer = run_inference(tc["prompt"], max_new_tokens=600)
    show_result("Code Generation", tc["id"], tc["prompt"], answer, time.time() - t0)

## 9 · Summary

In [ ]:
from IPython.display import display, Markdown

display(Markdown("""
## Results Summary

| Section | Tests |
|---------|-------|
| Summarization | 3 |
| Question Answering | 4 |
| Multiple Choice | 6 |
| Code Generation | 2 |
| **Total** | **15** |

### References
- [XMAiNframe on HuggingFace](https://huggingface.co/Fsoft-AIC/XMAiNframe-instruct-7b)
- [MainframeBench Dataset](https://huggingface.co/datasets/Fsoft-AIC/MainframeBench)
- [Paper: arXiv 2408.04660](https://arxiv.org/abs/2408.04660)
"""))